# 10 — MICOM Community Flux (mechanistic evidence)

**Decoupled run (CONTRACTS section 8).** Not part of `run_pipeline.py`: a community solve
takes about 10 minutes per sample over 277 samples. Run it here in the editor; every switch
is in the first code cell (`RUN_BATCH`, `N_THREADS`, `KEEP_MODELS`).

> **Not consumed yet.** NB08 as executed does not read this notebook's tables; the
> `mechanistic` evidence source takes effect with the NB08 rebuild (CONTRACTS section 6).
> Until then nothing in any `Tier` depends on NB10.

## One job

> **`tables/nb10_micom_species_flux_pairs_v2.csv`** — one row per (species, polyamine):
> the Spearman correlation, across samples, between the species' **secretion of the
> polyamine into the gut lumen** predicted by MICOM and the **measured** metabolite.
> Columns: `Species, Polyamine, Rho, QValue` (+ provenance and a coverage sensitivity).

Secretion is on the community scale: micom reports a taxon's exchange per gram of that
taxon, so it is multiplied by the taxon's abundance (the lumen balance holds only in that
form). It is still observational and abundance-driven — one source of several, never a proof.

## What micom does to the inputs (`micom_tools.py`, guarded by `checks.py` D8)

- Taxon ids come back rewritten (`Dialister hominis` → `Dialister_hominis`); they are mapped
  back to the species names every other notebook uses.
- **Some solves fail silently.** With the open-source hybrid solver (HiGHS + OSQP), the
  cooperative-tradeoff + pFBA step ends infeasible for some communities, and `grow()` returns
  that point without an error: community growth ~0, hundreds of spurious exchanges, a lumen
  that does not balance. Every sample is checked (`mt.solution_quality`); one that fails is
  recorded with the reason and left out — never read as zero flux.
- Without presolve, the solver did not finish one sample's first LP in 9 min once the diet
  was applied; every solve here presolves (`checks.py` E13).

## Fixes vs the earlier MICOM attempts (publication/NB09, Git notebook)

| Earlier problem | Here |
|---|---|
| AGORA103, ~42% abundance coverage | **AGORA2-GTDB** (`agora2_gtdb_species.qza`), GTDB-native names; median abundance coverage 64.6% |
| GTDB strains collapsed to NCBI binomials and summed | **no collapsing** — a taxon matches exactly or is reported unmatched |
| tradeoff hardcoded at 0.5 | **sweep** on 10 stratified, coverage-passing pilot samples; the largest tradeoff at which the most taxa grow |
| `strategy="none"` -> non-unique fluxes | **`strategy="pFBA"`** |
| growth rates discarded; fluxes >=80 deleted | growth rates kept; extreme fluxes **flagged, never deleted** |
| statistics on positive fluxes only (true by construction) | all solved samples, zeros included |
| HS->Early, MP->Advanced grouping | **v2.1 primary grouping** (MP excluded, HS quarantined) |

## Inputs / outputs

- **Inputs:** `Data/micom/agora2_gtdb_species.qza` (pre-extracted to
  `Data/micom/agora2_extracted/`), `Data/micom/western_diet_gut.csv`,
  `intermediate/preprocessed_data_v2.pkl` (full filtered species, raw relative abundance).
  `Data/micom/models_agora2/` (an older pre-built set) is **not used**: it was built from an
  earlier abundance table (coverage differs in all 277 samples, taxon counts in 91).
- **Checkpoints:** `intermediate/micom_v2/` — manifest, community models, tradeoff sweep,
  smoke result, batch chunks with their parameters, the samples not solved and why.
- **Tables:** `nb10_micom_sample_coverage_v2.csv`, `nb10_micom_community_flux_v2.csv`,
  `nb10_micom_species_flux_pairs_v2.csv`.

In [ ]:
import sys, os, json, hashlib, pickle, time, warnings, re
sys.path.insert(0, '.')
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

import utils
import micom_tools as mt

# Scoped, not blanket (checks.py E5): a bare ignore mutes convergence warnings.
warnings.filterwarnings('ignore', category=FutureWarning)

try:
    import micom
    from micom.workflows import build, grow
    from micom.qiime_formats import load_qiime_model_db
    print(f'micom {micom.__version__} loaded')
except ImportError as e:
    raise ImportError(
        'micom is required for NB10. Install the pinned versions into the pipeline venv:\n'
        '    pip install -r requirements-micom.txt\n'
        f'(import error: {e})')

# ---------------------------------------------------------------------------
# Run switches -- edit here. An environment variable of the same name overrides
# (MICOM_RUN_BATCH=1, MICOM_THREADS=6, MICOM_KEEP_MODELS=0) for scripted runs.
# ---------------------------------------------------------------------------
RUN_BATCH   = False   # True: after the smoke gate, run the full per-sample batch (hours)
N_THREADS   = None    # None: chosen below from CPU count and free memory
KEEP_MODELS = True    # False: delete the community models this notebook built, after the batch

def _env_flag(name, default):
    v = os.environ.get(name)
    return default if v is None else v.strip() not in ('0', '', 'false', 'False')

RUN_BATCH   = _env_flag('MICOM_RUN_BATCH', RUN_BATCH)
KEEP_MODELS = _env_flag('MICOM_KEEP_MODELS', KEEP_MODELS)
GB_PER_WORKER = 1.6   # measured: one micom worker holding a community model peaks at 1.3-1.6 GB
if os.environ.get('MICOM_THREADS'):
    N_THREADS = int(os.environ['MICOM_THREADS'])
if N_THREADS is None:
    _by_cpu = max(1, (os.cpu_count() or 2) // 2)
    _by_ram = max(1, int(psutil.virtual_memory().available / 1e9 // GB_PER_WORKER))
    N_THREADS = min(6, _by_cpu, _by_ram)
CHUNK = int(os.environ.get('MICOM_CHUNK', 2 * N_THREADS))   # samples per checkpoint

RESULTS_DIR = utils.RESULTS_DIR
DATA_DIR    = utils.DATA_DIR
TABLES      = RESULTS_DIR / 'tables'
FIGURES     = RESULTS_DIR / 'figures' / 'micom'
INTER       = RESULTS_DIR / 'intermediate'
MICOM_WORK  = INTER / 'micom_v2'                 # NB10's own checkpoint dir
for d in (TABLES, FIGURES, MICOM_WORK):
    d.mkdir(parents=True, exist_ok=True)

MICOM_DIR  = DATA_DIR / 'micom'
AGORA_DB   = MICOM_DIR / 'agora2_gtdb_species.qza'     # GTDB-native, section 8
# Section 8: western_diet_gut.csv names glucose EX_glc_e, which 0 of 1,439 AGORA2
# models carry (1,326 carry EX_glc_D(e)), so the largest carbon source in it was
# reaching nothing; and it supplies putrescine and spermidine at 300-5,000x the
# community's measured demand, which is why no taxon ever secretes them.
# v2/micom/diet_screen.py measured all three and wrote the corrected file, whose
# header records every change. Revert this line to compare against the old medium.
DIET_FILE  = Path('micom') / 'gut_diet_v2.csv'
MODELS_DIR = MICOM_WORK / 'models_agora2'              # community models built by THIS notebook
MODELS_DIR.mkdir(parents=True, exist_ok=True)

YACHIDA = utils.DATASET_DISCOVERY

ABUND_CUTOFF     = 1e-3   # drop taxa below 0.1% rel. abundance from the community
MIN_COVERAGE     = 0.50   # abundance coverage below this: flagged, left out of the sensitivity test
EXTREME_FLUX     = 80.0   # flag (never delete) |flux| above this
MIN_PAIR_SAMPLES = 20     # solved samples with a measurement needed to test a (species, polyamine)
STRATEGY         = 'pFBA'
SEED             = 42

missing = [str(p) for p in (AGORA_DB, DIET_FILE) if not p.exists()]
if missing:
    raise FileNotFoundError(
        'Required MICOM input(s) missing:\n  ' + '\n  '.join(missing) +
        '\nSee CONTRACTS.md section 8 for provenance.')
print(f'AGORA2 DB : {AGORA_DB.name} ({AGORA_DB.stat().st_size/1e6:.0f} MB)')
print(f'Diet      : {DIET_FILE.name}')
print(f'Work dir  : {MICOM_WORK}')
print(f'Threads   : {N_THREADS} (CPUs {os.cpu_count()}, free RAM '
      f'{psutil.virtual_memory().available/1e9:.1f} GB at {GB_PER_WORKER} GB/worker); '
      f'chunk {CHUNK} samples')
print(f'Batch run : {"ON" if RUN_BATCH else "OFF -- stops after the smoke gate; set RUN_BATCH = True above to run it"}')
print(f'Models    : {"kept" if KEEP_MODELS else "deleted"} after the batch')

In [ ]:
# ---------------------------------------------------------------------------
# Polyamine KEGG -> AGORA2/VMH metabolite ids, and the parser for the id forms micom
# writes ("ptrc[e]" on taxa, "ptrc_m" on the community, "EX_ptrc(e)" reactions), live in
# micom_tools so checks.py D8 can test them. A compound no model exchanges yields no
# mechanistic evidence for its pairs rather than a wrong number.
# ---------------------------------------------------------------------------
NAME_TO_KEGG = mt.NAME_TO_KEGG
print('Tracked polyamines (KEGG -> name -> VMH base ids):')
for k, (name, ids) in mt.POLYAMINE_VMH.items():
    print(f'  {k} {name:11s} <- {ids}')

In [ ]:
# ---------------------------------------------------------------------------
# Raw RELATIVE abundances (not CLR) for the FULL filtered species table, primary
# samples only. MICOM needs compositions, not log-ratios.
# ---------------------------------------------------------------------------
with open(INTER / 'preprocessed_data_v2.pkl', 'rb') as f:
    pdata = pickle.load(f)

# filtered_species = prevalence-filtered relative abundance, pre-CLR (CONTRACTS NB01).
filt = pdata['filtered_species'][YACHIDA].copy()
meta = pdata['harmonized_meta'][YACHIDA].copy()

# Measured metabolites: use the TRANSFORMED (log10) matrix; Spearman is rank-based
# so the monotone transform does not matter, and this matches NB02/NB08 scales.
mtb  = pdata['transformed_mtb'][YACHIDA].copy()
polyamine_columns = pdata.get('polyamine_columns', {}).get(YACHIDA, {})   # {kegg: column}
polyamine_proxies = pdata.get('polyamine_proxies', {}).get(YACHIDA, {})   # {kegg: column}
mtb_kegg_map      = pdata.get('mtb_kegg_map', {}).get(YACHIDA, {})        # {column: kegg}

for df in (filt, meta, mtb):
    if 'Sample' not in df.columns:
        df.reset_index(inplace=True)
        df.rename(columns={df.columns[0]: 'Sample'}, inplace=True)
    # Yachida ids are numeric (10021) in the pickle, but micom keys samples by string
    # (tax_long -> manifest -> exchanges). One id type everywhere: str. Mixed types
    # make merges raise and reindexes silently match nothing (all-zero flux).
    df['Sample'] = df['Sample'].astype(str)

# PRIMARY samples only (Healthy + Early_CRC + Advanced_CRC): HS/MP never reach MICOM.
prim = utils.primary_mask(meta.set_index('Sample')) if hasattr(utils, 'primary_mask') else None
if prim is not None:
    keep = meta.set_index('Sample').index[prim.values]
    meta = meta[meta['Sample'].isin(keep)].copy()
assert set(meta['Stage.Group'].dropna()) <= set(utils.STAGE_GROUP_ORDER), 'non-primary group reached NB10'

common = sorted(set(filt['Sample']) & set(meta['Sample']) & set(mtb['Sample']))
filt = filt.set_index('Sample').loc[common]
meta = meta.set_index('Sample').loc[common]
mtb  = mtb.set_index('Sample').loc[common]
print(f'Primary samples with species+metabolome: {len(common)}')
print(meta['Stage.Group'].value_counts().to_string())

# Renormalise each sample to a composition (sum=1).
species_cols = [c for c in filt.columns if c != 'Sample']
filt = filt[species_cols].div(filt[species_cols].sum(axis=1), axis=0).fillna(0.0)
print(f'Species in full filtered table: {filt.shape[1]}')

def resolve_metabolite_column(kegg_id):
    """Measured column for a polyamine, falling back to a pre-specified proxy."""
    for src in (polyamine_columns, polyamine_proxies):
        if kegg_id in src and src[kegg_id] in mtb.columns:
            return src[kegg_id], (src is polyamine_proxies)
    inv = {v: k for k, v in (mtb_kegg_map or {}).items()}
    if kegg_id in inv and inv[kegg_id] in mtb.columns:
        return inv[kegg_id], False
    for col in mtb.columns:
        if col.startswith(str(kegg_id) + '_'):
            return col, False
    return None, False

In [ ]:
# ---------------------------------------------------------------------------
# Build MICOM's taxonomy table. GTDB-native: a taxon matches AGORA2 exactly or is
# reported unmatched. We DO NOT strip _A suffixes or collapse strains (section 8).
# micom matches on BOTH the genus and species columns (adding its g__/s__ prefixes).
# ---------------------------------------------------------------------------
def gtdb_binomial(col: str):
    """Species binomial from a column name that may be a full GTDB lineage or a bare name.
    Returns (genus, species_full) keeping GTDB suffixes, or (None, None) if unusable."""
    s = str(col)
    if ';s__' in s or s.startswith('s__'):
        m = re.search(r's__([^;]+)', s)
        name = m.group(1).strip() if m else ''
    else:
        name = s.strip()
    toks = name.split()
    if len(toks) < 2:
        return (None, None)
    genus = toks[0]
    return (genus, name)

long = (filt.reset_index().melt(id_vars='Sample', var_name='_raw', value_name='abundance'))
long = long[long['abundance'] > ABUND_CUTOFF].copy()
gs = long['_raw'].map(gtdb_binomial)
long['genus']   = gs.map(lambda t: t[0])
long['species'] = gs.map(lambda t: t[1])
long = long.dropna(subset=['genus', 'species'])
long = long.rename(columns={'Sample': 'sample_id'})
long['sample_id'] = long['sample_id'].astype(str)
long['id'] = long['species']
tax_long = long[['sample_id', 'id', 'genus', 'species', 'abundance']].copy()
assert not tax_long.duplicated(['sample_id', 'species']).any(), 'a species appears twice in one sample'
print(f'tax_long: {tax_long["sample_id"].nunique()} samples, '
      f'{tax_long["species"].nunique()} unique species, {len(tax_long)} rows')

# micom rewrites every taxon id ("Dialister hominis" -> "Dialister_hominis"); this map
# turns the ids grow() returns back into the names NB02/NB03/NB08 use. It raises if two
# species would share one id.
TAXON_NAMES = mt.taxon_name_map(tax_long['species'])
print(f'micom taxon id -> species map: {len(TAXON_NAMES)} entries, no collisions')

In [ ]:
# ---------------------------------------------------------------------------
# >>> VERIFY (seconds, before the hours-long build): how many of our species
# actually exist in the AGORA2-GTDB db? Zero overlap = a naming problem to fix
# BEFORE build(), not after.
# ---------------------------------------------------------------------------
from zipfile import ZipFile
with ZipFile(str(AGORA_DB)) as z:
    man_files = [f for f in z.namelist() if f.endswith('manifest.csv')]
    if not man_files:
        raise RuntimeError(f'No manifest.csv inside {AGORA_DB.name}')
    with z.open(man_files[0]) as f:
        agora_mdf = pd.read_csv(f)
print(f'AGORA2 db manifest: {len(agora_mdf)} models')

# The db's species column carries the GTDB rank prefix ("s__Dialister invisus");
# micom.build strips it before matching, so this check must too, or it reports
# zero overlap while the build matches fine.
agora_species = set(agora_mdf['species'].astype(str).str.replace(r'^s__', '', regex=True))
ours = set(tax_long['species'])
hit  = ours & agora_species
print(f'Our species: {len(ours)} | AGORA2 species: {len(agora_species)} | matched: {len(hit)} '
      f'({100*len(hit)/max(1,len(ours)):.1f}% of ours)')
if len(hit) == 0:
    print('WARNING: zero overlap. Inspect the two namings below and adjust gtdb_binomial().')
    print('  ours  :', sorted(list(ours))[:5])
    print('  agora :', sorted(list(agora_species))[:5])
else:
    print('  matched examples:', sorted(list(hit))[:8])

In [ ]:
# ---------------------------------------------------------------------------
# Build per-sample community models from the pre-extracted AGORA2 dir (unzipping the
# .qza per sample would add ~1 min each). micom skips a sample whose pickle already
# exists, so an interrupted build resumes.
# ---------------------------------------------------------------------------
manifest_path = MICOM_WORK / 'manifest_agora2_v2.csv'
AGORA_EXTRACTED = MICOM_DIR / 'agora2_extracted'

# Sample ids are str throughout NB10 (nb10-03). The saved manifest's ids look numeric
# ("10021"), so read_csv must be told they are strings.
if (manifest_path.exists() and
        pd.read_csv(manifest_path, dtype={'sample_id': str}).shape[0] >= tax_long['sample_id'].nunique()):
    manifest = pd.read_csv(manifest_path, dtype={'sample_id': str})
    print(f'Reusing manifest ({len(manifest)} samples).')
else:
    if AGORA_EXTRACTED.exists() and any(AGORA_EXTRACTED.rglob('manifest.csv')):
        model_db = str(next(AGORA_EXTRACTED.rglob('manifest.csv')).parent)
        print(f'Reusing pre-extracted AGORA2 at {model_db}')
    else:
        AGORA_EXTRACTED.mkdir(parents=True, exist_ok=True)
        print('Pre-extracting AGORA2 .qza (one-time) ...')
        _m = load_qiime_model_db(str(AGORA_DB), str(AGORA_EXTRACTED))
        model_db = str(Path(_m['file'].iloc[0]).parent)
    print(f'Building community models (threads={N_THREADS}; no progress output until done) ...')
    manifest = build(tax_long, out_folder=str(MODELS_DIR), model_db=model_db,
                     cutoff=ABUND_CUTOFF, threads=N_THREADS)
    manifest.to_csv(manifest_path, index=False)
manifest['sample_id'] = manifest['sample_id'].astype(str)
print(f'Models available: {len(manifest)}')

# Reuse is only valid for models built from THIS taxonomy table and still on disk.
unknown = set(manifest['sample_id']) - set(meta.index)
assert not unknown, f'{len(unknown)} manifest sample ids not in meta, e.g. {sorted(unknown)[:3]}'
absent = [f for f in manifest['file'] if not (MODELS_DIR / str(f)).exists()]
if absent:
    raise FileNotFoundError(
        f'{len(absent)} model pickle(s) named in {manifest_path.name} are missing from '
        f'{MODELS_DIR} (e.g. {absent[:3]}) -- deleted after an earlier batch? Delete '
        f'{manifest_path.name} to rebuild them.')
n_taxa = tax_long.groupby('sample_id').size()
stale = manifest.loc[manifest['total_taxa'].astype(int).values
                     != n_taxa.reindex(manifest['sample_id']).values, 'sample_id']
if len(stale):
    raise ValueError(
        f'{len(stale)} model(s) were built from a different abundance table than this run\'s '
        f'(taxon counts differ, e.g. {list(stale[:3])}). Delete {manifest_path.name} and '
        f'the pickles in {MODELS_DIR} to rebuild.')

# Coverage per sample -> sample_coverage table (section 8: flagged, excluded in sensitivity).
# Abundance coverage (share of the community's abundance that has a model), as in
# CONTRACTS section 8 and micom's own <50% warning; the taxa-count fraction is kept
# alongside for reference.
cov = manifest[['sample_id']].copy()
cov['coverage'] = manifest['found_abundance_fraction']
cov['taxa_fraction'] = manifest['found_fraction']
cov['below_min_coverage'] = cov['coverage'] < MIN_COVERAGE
cov = cov.merge(meta.reset_index()[['Sample', 'Stage.Group']],
                left_on='sample_id', right_on='Sample', how='left').drop(columns='Sample')
cov.to_csv(TABLES / 'nb10_micom_sample_coverage_v2.csv', index=False)
print(f'Median AGORA2 abundance coverage: {cov["coverage"].median():.1%}; '
      f'{int(cov["below_min_coverage"].sum())} samples < {MIN_COVERAGE:.0%} (flagged).')

In [ ]:
# ---------------------------------------------------------------------------
# Choose the cooperative-tradeoff by a sweep on 10 stratified pilot samples
# (section 8), instead of hardcoding 0.5. Value + full sweep are saved.
# Needs a QP-capable solver (osqp / HiGHS); community.py is stock micom.
# ---------------------------------------------------------------------------
tradeoff_path = MICOM_WORK / 'tradeoff_sweep_v2.csv'
N_PILOT = 10
medium = pd.read_csv(DIET_FILE, comment='#')   # the diet file carries its reasoning in a # header
if not {'reaction', 'flux'} <= set(medium.columns):
    raise ValueError(f'{DIET_FILE.name} needs reaction,flux columns; got {list(medium.columns)}')
# AGORA community medium reactions are lumen exchanges (_m); convert if the file uses _e.
n_e = medium['reaction'].str.endswith('_e').sum()
medium['reaction'] = medium['reaction'].str.replace(r'_e$', '_m', regex=True)
dup = medium[medium['reaction'].duplicated(keep=False)]
if dup.groupby('reaction')['flux'].nunique().gt(1).any():
    raise ValueError(f'{DIET_FILE.name} lists a reaction twice with different fluxes:\n{dup}')
medium = medium.drop_duplicates('reaction')
print(f'Diet: {len(medium)} medium reactions ({n_e} converted _e->_m; identical duplicates '
      f'dropped: {sorted(dup["reaction"].unique()) or "none"}).')
_pa_diet = medium[medium['reaction'].map(lambda r: mt.classify_metabolite(r)[0] is not None)]
print('Diet supplies tracked compounds (max uptake): '
      + ', '.join(f'{mt.classify_metabolite(r)[1]} {f:g}' for r, f in
                  zip(_pa_diet['reaction'], _pa_diet['flux'])))

if tradeoff_path.exists():
    sweep = pd.read_csv(tradeoff_path, dtype={'sample_id': str})
    print(f'Reusing tradeoff sweep ({len(sweep)} rows, {sweep["sample_id"].nunique()} pilot samples).')
else:
    # Pilot = samples the analysis actually rests on (coverage >= MIN_COVERAGE), drawn
    # round-robin across stage groups so 10 splits 4/3/3. Taking 4 per group and then
    # truncating to 10 left Healthy with 2 and, by chance, 7 of 10 below 50% coverage.
    ok_cov = cov.loc[~cov['below_min_coverage'], ['sample_id', 'Stage.Group']]
    shuffled = [g.sample(frac=1, random_state=SEED)['sample_id'].tolist()
                for _, g in ok_cov.groupby('Stage.Group')]
    pilot_ids = [s for rank in zip(*shuffled) for s in rank][:N_PILOT]
    pilot_man = manifest[manifest['sample_id'].isin(pilot_ids)]
    print(f'Tradeoff pilot on {len(pilot_man)} samples (coverage >= {MIN_COVERAGE:.0%}): '
          + ', '.join(f'{s} ({g})' for s, g in
                      ok_cov.set_index('sample_id').loc[pilot_ids, 'Stage.Group'].items()))
    print('No progress output until done (about 1 h per sample, N_THREADS at a time).')
    # presolve=True is required with the diet applied: without it the hybrid
    # HiGHS/OSQP solver did not finish even the initial LP of one pilot sample in
    # 9 min (the first sweep ran >22 h on one sample); with it, the LP takes ~40 s
    # and each tradeoff value ~6 min per sample. The batch grow() also presolves.
    from micom.workflows import tradeoff as micom_tradeoff
    sweep = micom_tradeoff(pilot_man, model_folder=str(MODELS_DIR), medium=medium,
                           tradeoffs=np.arange(0.1, 1.01, 0.1), threads=N_THREADS,
                           presolve=True)
    sweep.to_csv(tradeoff_path, index=False)

# micom's rule (mt.choose_tradeoff, checks.py D8): the LARGEST tradeoff that still lets
# the most taxa grow. Low tradeoffs tie at "all taxa grow" because they cap community
# growth at that fraction of its maximum; the first tie (0.1) held every community to
# 10% of its achievable growth.
OPT_TRADEOFF, tradeoff_stats = mt.choose_tradeoff(sweep)
print(tradeoff_stats.round(4).to_string())
print(f'Chosen tradeoff: {OPT_TRADEOFF:.2f}')

In [ ]:
# ---------------------------------------------------------------------------
# Smoke gate: one sample, exactly as the batch solves it, before hours of compute.
# Stops here if the solution is not a real one (mt.solution_quality: community growth and
# the lumen balance over every metabolite); reports time, memory and which taxa SECRETE
# each polyamine (community-level rows only say what the diet supplies).
# ---------------------------------------------------------------------------
# Keyed by tradeoff, so a smoke run at a different tradeoff is never reused.
smoke_path = MICOM_WORK / f'smoke_one_v2_t{OPT_TRADEOFF:.2f}.pkl'
one = manifest.iloc[[0]]
if smoke_path.exists():
    with open(smoke_path, 'rb') as f:
        smoke = pickle.load(f)
    smoke_secs = None
    print(f'Smoke result for tradeoff {OPT_TRADEOFF:.2f} reused ({smoke_path.name}).')
else:
    print(f'Smoke grow() on sample {one["sample_id"].iloc[0]} ({STRATEGY}, tradeoff {OPT_TRADEOFF:.2f}) ...')
    import tracemalloc
    tracemalloc.start(); t0 = time.time()
    smoke = grow(one, model_folder=str(MODELS_DIR), medium=medium,
                 tradeoff=OPT_TRADEOFF, threads=1, strategy=STRATEGY, presolve=True)
    smoke_secs = time.time() - t0; _, peak = tracemalloc.get_traced_memory(); tracemalloc.stop()
    with open(smoke_path, 'wb') as f:
        pickle.dump(smoke, f)
    print(f'Smoke solved: {len(smoke.exchanges)} exchange rows, {smoke_secs:.0f}s, '
          f'~{peak/1e9:.1f} GB python heap.')

sx = smoke.exchanges.copy()
sx['sample_id'] = sx['sample_id'].astype(str)
assert not mt.missing_samples(one['sample_id'], sx), 'the smoke sample returned no solution'
sq = mt.solution_quality(sx, smoke.growth_rates).iloc[0]
assert sq['valid'], (f'the smoke sample\'s solution is not a real one (community growth '
                     f'{sq["community_growth"]:.3g}, lumen balance off by {sq["max_balance_residual"]:.3g})')
s_pa = mt.polyamine_exchanges(sx, TAXON_NAMES)        # raises if a taxon id has no species name
s_bal = mt.community_balance(sx, s_pa)
print(f'Smoke solution valid: community growth {sq["community_growth"]:.3f}/h, lumen balance '
      f'within {sq["max_balance_residual"]:.1e}.')
print('Smoke sample, per polyamine (community scale, mmol/gDW/h; net < 0 = taken from the diet):')
print(s_bal.set_index('Polyamine')[['Net_medium_flux', 'Gross_export', 'Gross_uptake',
                                    'N_exporting_taxa', 'N_uptaking_taxa']].round(5).to_string())
silent = sorted(set(mt.NAME_TO_KEGG) - set(s_bal.loc[s_bal['N_exporting_taxa'] > 0, 'Polyamine']))
if silent:
    print(f'WARNING: no taxon secretes {", ".join(silent)} in this sample; where that holds across '
          f'samples, those compounds get no species-level export to correlate.')
if smoke_secs is not None:
    print(f'Batch estimate: {len(manifest)} samples x {smoke_secs/60:.0f} min / {N_THREADS} workers '
          f'= ~{len(manifest) * smoke_secs / N_THREADS / 3600:.1f} h; needs ~{N_THREADS * GB_PER_WORKER:.0f} GB '
          f'RAM (free now: {psutil.virtual_memory().available/1e9:.1f} GB).')

In [ ]:
# ---------------------------------------------------------------------------
# Full batch, checkpointed per chunk so an interruption resumes instead of restarting.
# The parameters the chunks were solved with are recorded; results from different
# parameters are never mixed. Every sample's solution is then checked
# (mt.solution_quality): micom returns an INFEASIBLE tradeoff/pFBA point without an
# error, so a sample counts as solved only if its community grows and its lumen balances.
# A sample that is not solved is recorded with the reason and left out downstream
# (never read as zero flux).
# ---------------------------------------------------------------------------
from cobra.util.solver import OptimizationError

class BatchOff(Exception):
    """Raised on purpose: RUN_BATCH is False and samples remain to be solved."""

def _sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

BATCH_PARAMS = {'tradeoff': round(OPT_TRADEOFF, 4), 'strategy': STRATEGY, 'presolve': True,
                'diet_sha256': _sha256(DIET_FILE), 'manifest_sha256': _sha256(manifest_path),
                'models_dir': str(MODELS_DIR), 'micom': micom.__version__}
params_path = MICOM_WORK / 'batch_params_v2.json'
failed_path = MICOM_WORK / 'batch_not_solved_v2.csv'
exch_chunks = sorted(MICOM_WORK.glob('chunk_exchanges_*.pkl'))
if params_path.exists():
    old = json.loads(params_path.read_text())
    if old != BATCH_PARAMS:
        diff = {k: (old.get(k), v) for k, v in BATCH_PARAMS.items() if old.get(k) != v}
        raise ValueError(f'Batch checkpoints in {MICOM_WORK} were solved with other parameters '
                         f'(old, new): {diff}. Move the chunk_*.pkl files and {params_path.name} '
                         'aside to start over.')
elif exch_chunks:
    raise ValueError(f'{len(exch_chunks)} batch chunk(s) but no {params_path.name}: their parameters '
                     'are unknown. Move them aside to start over.')

parts, done = [], set()
for p in exch_chunks:
    d = pd.read_pickle(p)
    d['sample_id'] = d['sample_id'].astype(str)
    parts.append(d)
    done |= set(d['sample_id'])
want = list(manifest['sample_id'])
todo = [s for s in want if s not in done]
print(f'Batch: {len(done)} of {len(want)} samples returned by micom in earlier runs, {len(todo)} to go.')

if todo and not RUN_BATCH:
    eta = (f', about {len(todo) * smoke_secs / N_THREADS / 3600:.1f} h with {N_THREADS} workers'
           if smoke_secs else '')
    raise BatchOff(f'stopping after the smoke gate on purpose: RUN_BATCH is False. To run the '
                   f'{len(todo)} remaining samples{eta}, set RUN_BATCH = True in nb10-01 and re-run '
                   'nb10-01 and this cell. Checkpoints resume.')

if todo:
    params_path.write_text(json.dumps(BATCH_PARAMS, indent=1))
    stamp = datetime.now().strftime('%Y%m%d-%H%M%S')    # unique chunk names: a resume never overwrites
    t_start, n_bad = time.time(), 0
    print(f'[{datetime.now():%H:%M}] solving {len(todo)} samples, {CHUNK} per checkpoint, '
          f'{N_THREADS} workers ...', flush=True)
    for n, i in enumerate(range(0, len(todo), CHUNK)):
        ids = todo[i:i + CHUNK]
        sub = manifest[manifest['sample_id'].isin(ids)]
        try:
            gr = grow(sub, model_folder=str(MODELS_DIR), medium=medium, tradeoff=OPT_TRADEOFF,
                      threads=N_THREADS, strategy=STRATEGY, presolve=True)
        except OptimizationError as e:   # micom raises only when EVERY sample of the call failed
            raise RuntimeError(f'no sample of chunk {n + 1} ({ids[:3]} ...) solved: {e}') from e
        ex = gr.exchanges.copy()
        ex['sample_id'] = ex['sample_id'].astype(str)
        gr.growth_rates.to_pickle(MICOM_WORK / f'chunk_growth_{stamp}_{n:03d}.pkl')
        ex.to_pickle(MICOM_WORK / f'chunk_exchanges_{stamp}_{n:03d}.pkl')
        parts.append(ex)
        q = mt.solution_quality(ex, gr.growth_rates)
        bad = sorted(set(ids) - set(q.loc[q['valid'], 'sample_id']))
        n_bad += len(bad)
        n_done, el = i + len(ids), time.time() - t_start
        print(f'  [{datetime.now():%H:%M}] chunk {n + 1}: {n_done}/{len(todo)} samples'
              + (f', NOT SOLVED {bad}' if bad else '')
              + f' ({n_bad} so far); elapsed {el / 60:.0f} min, '
              f'ETA {el / n_done * (len(todo) - n_done) / 60:.0f} min', flush=True)

exchanges = pd.concat(parts, ignore_index=True)
growth_all = pd.concat([pd.read_pickle(p) for p in sorted(MICOM_WORK.glob('chunk_growth_*.pkl'))],
                       ignore_index=True)
growth_all['sample_id'] = growth_all['sample_id'].astype(str)
assert not growth_all.duplicated(['sample_id', 'taxon']).any(), 'a sample was solved twice across chunks'

quality = mt.solution_quality(exchanges, growth_all)
SOLVED = sorted(quality.loc[quality['valid'], 'sample_id'])
status = pd.DataFrame({'sample_id': want}).merge(quality, on='sample_id', how='left')
status['reason'] = np.where(status['community_growth'].isna(), 'no solution returned',
                   np.where(status['valid'].fillna(False).astype(bool), '',
                            'infeasible solution returned without an error'))
not_solved = status[status['reason'] != '']
not_solved.to_csv(failed_path, index=False)
exchanges = exchanges[exchanges['sample_id'].isin(SOLVED)].reset_index(drop=True)
growth_all = growth_all[growth_all['sample_id'].isin(SOLVED)].reset_index(drop=True)
print(f'Solved {len(SOLVED)}/{len(want)} samples. Not solved: {len(not_solved)} '
      f'{not_solved["reason"].value_counts().to_dict() or ""} -> {failed_path.name}')
if len(not_solved):
    print('  (a returned-but-infeasible sample is not retried: the same settings fail the same way)')
assert set(SOLVED) <= set(mtb.index), 'a solved sample has no metabolome row'

# Flag (do not delete) extreme fluxes.
exchanges['extreme'] = exchanges['flux'].abs() > EXTREME_FLUX
print(f'Extreme-flux rows flagged (|flux|>{EXTREME_FLUX}): {int(exchanges["extreme"].sum())}')

In [ ]:
# ---------------------------------------------------------------------------
# Per (sample, species, polyamine) SECRETION into the lumen, on the community scale.
# micom reports a taxon's exchange per gram of that taxon; abundance * flux is its
# contribution to the lumen, and only that form satisfies the lumen balance
#     community (medium) flux = sum over taxa of abundance * taxon flux,
# which every solved sample met in nb10-09 and is re-asserted here for the polyamines.
# Taxon ids are mapped back to species names (micom rewrote them).
# ---------------------------------------------------------------------------
pa_ex = mt.polyamine_exchanges(exchanges, TAXON_NAMES)   # raises on an unmapped taxon
sp_flux = mt.species_export(pa_ex)
comm = mt.community_balance(exchanges, pa_ex)
bal_tol = mt.BALANCE_TOL_FACTOR * float(exchanges['tolerance'].max())
bal_err = (comm['Net_medium_flux'] - (comm['Gross_export'] - comm['Gross_uptake'])).abs().max()
assert bal_err <= bal_tol, f'lumen balance off by {bal_err:g} (> {bal_tol:g}): flux units not as assumed'
comm.to_csv(TABLES / 'nb10_micom_community_flux_v2.csv', index=False)
print(f'Saved: nb10_micom_community_flux_v2.csv ({len(comm)} sample x polyamine rows over '
      f'{len(SOLVED)} solved samples; lumen balance holds to {bal_err:.1e})')

# What the model says, per compound: is it secreted by any taxon, and is the community a
# net producer (net medium flux > 0) or a net consumer of the diet's supply?
summary = (comm.groupby('Polyamine')
               .agg(samples=('sample_id', 'nunique'),
                    with_secretion=('N_exporting_taxa', lambda v: int((v > 0).sum())),
                    net_producer=('Net_medium_flux', lambda v: int((v > bal_tol).sum())),
                    median_net_flux=('Net_medium_flux', 'median')))
summary['species_secreting'] = sp_flux.groupby('Polyamine')['Species'].nunique()
summary = summary.reindex(sorted(mt.NAME_TO_KEGG)).fillna({'samples': 0, 'with_secretion': 0,
                                                            'net_producer': 0, 'species_secreting': 0})
print(f'Per compound over {len(SOLVED)} solved samples:')
print(summary.to_string())

In [ ]:
# ---------------------------------------------------------------------------
# For each (species, polyamine): Spearman(species secretion, measured metabolite) across
# SOLVED samples. A solved sample where the species is absent or does not secrete counts
# as 0; a sample micom could not solve is left out. BH across all tested pairs, one
# family; the same test on samples with abundance coverage >= MIN_COVERAGE is the
# pre-declared sensitivity analysis (CONTRACTS section 8), with its own BH family.
# ---------------------------------------------------------------------------
meas, proxy_flag = {}, {}
for name, kegg in NAME_TO_KEGG.items():
    col, is_proxy = resolve_metabolite_column(kegg)
    if col is not None:
        meas[name] = mtb[col]
        proxy_flag[name] = is_proxy
print('Measured polyamines available:', sorted(meas))

cov_ok = set(cov.loc[~cov['below_min_coverage'], 'sample_id'])
present = (growth_all.assign(Species=growth_all['taxon'].astype(str).map(TAXON_NAMES))
                     .groupby('Species')['sample_id'].nunique())

def _spearman(x, y):
    if len(x) < MIN_PAIR_SAMPLES or x.nunique() < 3:
        return np.nan, np.nan
    r, p = spearmanr(x, y)
    return (r, p) if np.isfinite(r) else (np.nan, np.nan)

rows, skipped = [], {}
for (species, pa_name), g in sp_flux.groupby(['Species', 'Polyamine']):
    if pa_name not in meas:
        skipped['compound not measured'] = skipped.get('compound not measured', 0) + 1
        continue
    x = g.set_index('sample_id')['export_community'].reindex(SOLVED).fillna(0.0)
    y = meas[pa_name].reindex(SOLVED)
    ok = y.notna().values
    rho, p = _spearman(x[ok], y[ok])
    if np.isnan(rho):
        skipped['secreted in too few samples to rank'] = skipped.get('secreted in too few samples to rank', 0) + 1
        continue
    ok_c = ok & x.index.isin(cov_ok)
    rho_c, p_c = _spearman(x[ok_c], y[ok_c])
    rows.append({
        'Species': species, 'Polyamine': pa_name, 'KEGG': NAME_TO_KEGG[pa_name],
        'Rho': rho, 'PValue': p, 'N_samples': int(ok.sum()),
        'N_present': int(present.get(species, 0)), 'N_secreting': int((x[ok] > 0).sum()),
        'Mean_export_flux': float(x[ok].mean()),
        'Frac_samples_producing': float((x[ok] > 0).mean()),
        'Rho_cov_ok': rho_c, 'PValue_cov_ok': p_c, 'N_samples_cov_ok': int(ok_c.sum()),
        'Measured_is_proxy': bool(proxy_flag.get(pa_name, False)),
        'MICOM_status': 'computed',
    })

cols = ['Species', 'Polyamine', 'KEGG', 'Rho', 'PValue', 'QValue', 'N_samples', 'N_present',
        'N_secreting', 'Mean_export_flux', 'Frac_samples_producing', 'Rho_cov_ok', 'PValue_cov_ok',
        'QValue_cov_ok', 'N_samples_cov_ok', 'Measured_is_proxy', 'MICOM_status']
pairs = pd.DataFrame(rows, columns=[c for c in cols if not c.startswith('QValue')])
pairs['QValue'] = multipletests(pairs['PValue'], method='fdr_bh')[1] if len(pairs) else []
pairs['QValue_cov_ok'] = np.nan
_c = pairs['PValue_cov_ok'].notna()
if _c.any():
    pairs.loc[_c, 'QValue_cov_ok'] = multipletests(pairs.loc[_c, 'PValue_cov_ok'], method='fdr_bh')[1]
pairs = pairs[cols].sort_values(['QValue', 'Rho'], ascending=[True, False]).reset_index(drop=True)
pairs.to_csv(TABLES / 'nb10_micom_species_flux_pairs_v2.csv', index=False)

print(f'Saved: nb10_micom_species_flux_pairs_v2.csv ({len(pairs)} species-polyamine pairs tested; '
      f'not tested: {skipped or "none"})')
sig = pairs[(pairs['QValue'] < utils.FDR_THRESHOLD) & (pairs['Rho'] > 0)]
per = pd.DataFrame({
    'species_secreting': sp_flux.groupby('Polyamine')['Species'].nunique(),
    'pairs_tested': pairs.groupby('Polyamine').size(),
    f'positive_q<{utils.FDR_THRESHOLD}': sig.groupby('Polyamine').size(),
}).reindex(sorted(NAME_TO_KEGG)).fillna(0).astype(int)
print(per.to_string())
if len(pairs):
    print(pairs.head(15)[['Species', 'Polyamine', 'Rho', 'QValue', 'N_secreting', 'N_present',
                          'Rho_cov_ok', 'QValue_cov_ok']].round(4).to_string(index=False))

In [ ]:
# ---------------------------------------------------------------------------
# Models are large (~28 GB for 277 samples) but took ~10 h to build; they are kept unless
# KEEP_MODELS is False, so a re-run with another tradeoff or strategy needs no rebuild.
# Only the models THIS notebook built (MODELS_DIR) are ever deleted -- never the older
# pre-built set in Data/micom/models_agora2.
# ---------------------------------------------------------------------------
if not KEEP_MODELS:
    assert MODELS_DIR.resolve().is_relative_to(MICOM_WORK.resolve()), \
        f'refusing to delete models outside {MICOM_WORK}'
    n = 0
    for p in MODELS_DIR.glob('*.pickle'):
        p.unlink(); n += 1
    manifest_path.unlink(missing_ok=True)     # its models are gone; a re-run must rebuild
    print(f'Deleted {n} community model pickles and {manifest_path.name}.')
else:
    print(f'Community models kept in {MODELS_DIR} (KEEP_MODELS = True).')

print('\nNB10 done. Outputs:')
for t in ('nb10_micom_sample_coverage_v2.csv', 'nb10_micom_community_flux_v2.csv',
          'nb10_micom_species_flux_pairs_v2.csv'):
    p = TABLES / t
    print(f'  {"OK " if p.exists() else "MISSING "} {t}')
print('\nNB08 as executed does not read these tables yet; the mechanistic evidence source takes\n'
      'effect with the NB08 rebuild (CONTRACTS section 6).')